# DataMend: STNet jobs on Kaggle (one notebook per account)

Trains and evaluates the STNet fault detector (`ml-service/qc/stnet.py`) on the RF25 validation benchmark. SYNTHETIC faults only. Built to be split across several accounts: each account runs its own job(s), then the small results are merged.

**Suggested split over 4 accounts** (same schedule for every job, so the comparison is fair):

| account | `RUN_JOBS` | what it answers |
|---|---|---|
| A | `['main']` | the main run (seed 42) plus fusion, ceiling and the out-of-sample false-alarm check |
| B | `['hold']` | held-out fault kinds (zeroed, scale, drift removed from training): does it generalise? |
| C | `['nomem']` | no short- or long-range memory: does memory matter? |
| D | `['seed43']` | a second seed of the main run: how much is seed noise? (or `['norobust']`, `['nodisc']`) |

With two GPUs (T4 x2) one account can run two jobs at once, for example `['hold', 'nomem']`.

**Before you run it**
1. Settings: **Accelerator** = GPU, **Internet** = On.
2. **Add Input**: attach the RF25 dataset (the link is already in `DATASET_LINK`; it is public).
3. If the GitHub repository is private, add a Kaggle secret `GITHUB_TOKEN` (Add-ons > Secrets): a token with read access to the repository. Never paste a token into a cell.
4. Set `RUN_JOBS` in the next cell. Then Run All, or Save Version (Commit) to run it in the background.

Results: `stnet_results_<jobs>.zip` in the output. Unzip it inside `ml-service/` of your checkout and commit. If a job stops at the time cap, Save Version, attach that output as an input, put its link in `RESUME_LINK` and run again: it resumes from the last year-step.

Kaggle's terms allow one account per person. Use accounts that belong to the people running them.

In [ ]:
# ------------------------------- edit these -------------------------------
RUN_JOBS = ['main']          # what THIS notebook runs, one job per GPU at the same time: main, seed43, hold, nomem, norobust, nodisc
DATASET_LINK = '/kaggle/input/datasets/nathashalewis/majorproj-rainfall'
REPO_URL = 'https://github.com/NL0711/datamend.git'
BRANCH = 'main'
RUN_NAME = 'base'
EPOCHS, STEPS, LONG_STEPS = 4, 200, 30      # the same for every job: comparing jobs needs the same schedule
TIME_CAP_HOURS = 8.0         # per job; a job that reaches it stops cleanly and can be resumed
RESUME_LINK = ''             # link of a previous version's output (attached as input) to continue from
PUSH_RESULTS = False         # True: also commit the small results to branch kaggle-results-<jobs> (needs a GITHUB_TOKEN secret with write access)
EXPECTED_BENCH_HASH = 'c37de7e3cf2be90e'    # the frozen benchmark every result so far was scored on; a different hash stops the run
JOBS = {
    'main':     dict(prefix='stnet',          seed=42, args=[], post=True),
    'seed43':   dict(prefix='stnet',          seed=43, args=[], post=False),
    'hold':     dict(prefix='stnet_hold',     seed=42, args=['--holdout-kinds', 'zeroed', 'scale', 'drift'], post=False),
    'nomem':    dict(prefix='stnet_nomem',    seed=42, args=['--no-gru', '--no-long'], post=False),
    'norobust': dict(prefix='stnet_norobust', seed=42, args=['--no-robust'], post=False),
    'nodisc':   dict(prefix='stnet_nodisc',   seed=42, args=['--no-disc'], post=False),
}
unknown = [j for j in RUN_JOBS if j not in JOBS]
if unknown: raise ValueError('unknown job(s): ' + ', '.join(unknown) + '. Choose from ' + ', '.join(JOBS))

In [ ]:
import os, sys, subprocess, pathlib, shutil, json, time, importlib
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').is_dir() else pathlib.Path.cwd()
SECRET = ''
try:
    from kaggle_secrets import UserSecretsClient
    SECRET = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as e:
    print('No GITHUB_TOKEN secret (fine for a public repository):', type(e).__name__)
def clean(t): return t.replace(SECRET, '***') if SECRET else t
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = clean((r.stdout + r.stderr).strip())
    if show: print('$', clean(cmd), chr(10) + out[-1200:])
    if r.returncode: raise RuntimeError('failed: ' + clean(cmd)[:120] + chr(10) + out[-1200:])
    return out
def auth_url(url, token): return url.replace('https://', 'https://x-access-token:' + token + '@') if token else url
os.environ['GIT_TERMINAL_PROMPT'] = '0'
REPO = WORK / 'datamend'
url = auth_url(REPO_URL, SECRET)
try:
    if (REPO / '.git').is_dir():
        sh('git -C ' + str(REPO) + ' fetch ' + url + ' ' + BRANCH + ' && git -C ' + str(REPO) + ' checkout -B ' + BRANCH + ' FETCH_HEAD')
    else:
        sh('git clone --branch ' + BRANCH + ' ' + url + ' ' + str(REPO))
        sh('git -C ' + str(REPO) + ' remote set-url origin ' + REPO_URL, show=False)
except RuntimeError as e:
    raise RuntimeError('Could not get the repository. Check that Internet is ON in the notebook settings and, if the repository is private, that the GITHUB_TOKEN secret is attached to this notebook.') from None
sh('git -C ' + str(REPO) + ' log -1 --oneline')
ML = REPO / 'ml-service'
os.chdir(ML)
sys.path.insert(0, str(ML))
for _m in [k for k in sys.modules if k == 'qc' or k.startswith('qc.')]:
    del sys.modules[_m]
importlib.invalidate_caches()
sh('pip install -q numpy pandas scipy scikit-learn xarray netCDF4 matplotlib pyyaml pyarrow kagglehub psutil nvidia-ml-py', show=False)
print('repo:', REPO, '| cwd:', os.getcwd())

In [ ]:
import torch
from qc.kaggle_data import locate_rf25, find_saved_outputs
from qc.config import load_config, year_range

cfg0 = load_config(ML / 'qc' / 'configs' / 'base.yaml', [])
need = sum(len(year_range(cfg0.years[k])) for k in ('train', 'val', 'test', 'stress') if cfg0.years.get(k))
def _download(slug):
    import kagglehub
    return kagglehub.dataset_download(slug)
RF25_DIR = locate_rf25(DATASET_LINK, expected_files=need, download=_download)
n_gpu = torch.cuda.device_count()
print('RF25:', RF25_DIR, '|', len(list(RF25_DIR.glob('RF25_ind*_rfp25.nc'))), 'files | GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
if n_gpu == 0: raise RuntimeError('No GPU: turn on a GPU accelerator in the notebook settings.')
if len(RUN_JOBS) > n_gpu: raise RuntimeError('RUN_JOBS has ' + str(len(RUN_JOBS)) + ' jobs but there are ' + str(n_gpu) + ' GPU(s): one job per GPU.')
os.environ['DATAMEND_RF25_DIR'] = str(RF25_DIR)
os.environ['DATAMEND_DATA_DIR'] = str(WORK / 'data')
OUT_ROOT = WORK / 'artifacts'
RUN = RUN_NAME
BASE = ['--profile', 'single-gpu', '--set', 'name=' + RUN]

# Continue from a previous session (optional)
if RESUME_LINK.strip():
    prev = find_saved_outputs(RESUME_LINK)
    if prev is None:
        raise RuntimeError('no previous output (a folder with artifacts/qc) found for ' + RESUME_LINK + '. Attach the previous version output as an input first.')
    for sub in ('artifacts', 'data'):
        if (prev / sub).is_dir():
            shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
    print('restored from', prev)
else:
    print('fresh run (no RESUME_LINK)')

In [ ]:
def run_stream(step, *args):
    cmd = [sys.executable, '-u', '-m', 'qc', step, *args, *BASE]
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ML)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError('step failed: ' + step)

# Shared, cheap, identical on every account: data check and splits, the frozen fault benchmark, the untrained baselines (rules are needed for the fusion).
run_stream('prepare')
run_stream('benchmark')
run_stream('untrained')

from qc.benchmark import bench_dir
cfg = load_config(ML / 'qc' / 'configs' / 'base.yaml', ['name=' + RUN])
bench_hash = json.loads((bench_dir(cfg) / 'manifest.json').read_text())['hash']
print('benchmark hash:', bench_hash)
if bench_hash != EXPECTED_BENCH_HASH:
    raise RuntimeError('benchmark hash ' + bench_hash + ' is not the expected ' + EXPECTED_BENCH_HASH + ': results from this notebook would not be comparable with the earlier ones. Stop and check the repository version.')

In [ ]:
# One process per job, each on its own GPU. Logs go to /kaggle/working/job_<name>.log
procs = []
for i, name in enumerate(RUN_JOBS):
    j = JOBS[name]
    cmd = [sys.executable, '-u', '-m', 'qc', 'stnet', '--seeds', str(j['seed']), '--prefix', j['prefix'], '--epochs', str(EPOCHS), '--steps', str(STEPS),
           '--long-steps', str(LONG_STEPS), '--time-cap-hours', str(TIME_CAP_HOURS), *j['args'], *BASE, '--set', 'compute.devices=[cuda:' + str(i) + ']']
    logf = open(WORK / ('job_' + name + '.log'), 'a')
    procs.append((name, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, text=True, cwd=ML), logf))
    print('started', name, 'on cuda:' + str(i), '|', ' '.join(cmd[4:]))

done = {}
while len(done) < len(procs):
    time.sleep(60)
    for name, p, logf in procs:
        if name in done:
            continue
        rc = p.poll()
        try:
            tail = (WORK / ('job_' + name + '.log')).read_text().strip().splitlines()[-1][:150]
        except Exception:
            tail = ''
        print(time.strftime('%H:%M'), name, 'running' if rc is None else 'finished rc=' + str(rc), '|', tail)
        if rc is not None:
            done[name] = rc
            logf.close()
bad = [n for n, rc in done.items() if rc]
if bad:
    raise RuntimeError('job(s) failed: ' + ', '.join(bad) + '. Read /kaggle/working/job_<name>.log (the last lines say what went wrong).')
unfinished = [n for n in RUN_JOBS if 'stopped at the time cap' in (WORK / ('job_' + n + '.log')).read_text().splitlines()[-3:].__str__()]
if unfinished:
    print('STOPPED AT THE TIME CAP:', unfinished, '- Save Version, attach the output as an input, set RESUME_LINK and run again. The post steps below are skipped for these.')

In [ ]:
# Fusion with the untrained rules, the per-kind ceiling check and the out-of-sample false-alarm check, for the main job only.
for name in RUN_JOBS:
    j = JOBS[name]
    if not j['post'] or name in unfinished:
        continue
    s, pre = str(j['seed']), j['prefix']
    run_stream('ablate', '--seed', s, '--tag', pre, '--subset', f'rules+{pre}_fault', '--subset', f'rules+{pre}_fault+{pre}_clean',
               '--subset', f'rules+{pre}_fault+{pre}_zeroed+{pre}_scale+{pre}_drift')
    run_stream('ceiling', '--seed', s, '--tag', pre, '--components', *[f'{pre}_{k}_s{s}' for k in ('fault', 'clean', 'disc', 'zeroed', 'scale', 'drift')])
    run_stream('calibrate', '--seed', s, '--components', f'{pre}_fault_s{s}')

In [ ]:
# Package the small results: tables, json reports, curve images, telemetry. No checkpoints, no score arrays.
import zipfile
run_dir = OUT_ROOT / 'qc' / RUN
keep = []
for pat in ('**/*.md', '**/*.json', '**/*.png'):
    keep += [p for p in run_dir.glob(pat) if p.is_file() and p.stat().st_size < 50e6 and 'scores' not in p.parts and 'checkpoints' not in p.parts]
keep = list(dict.fromkeys(keep))
zname = WORK / ('stnet_results_' + '_'.join(RUN_JOBS) + '.zip')
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, p.relative_to(WORK))
print(zname, '(' + format(zname.stat().st_size / 1e6, '.1f') + ' MB,', len(keep), 'files)')
print('Unzip it inside ml-service/ of your checkout (it holds artifacts/qc/' + RUN + '/...), then commit and push.')

if PUSH_RESULTS:
    if not SECRET:
        print('PUSH_RESULTS skipped: no GITHUB_TOKEN secret (it needs write access).')
    else:
        try:
            from qc.kaggle_sync import sync_results
            branch = 'kaggle-results-' + '-'.join(RUN_JOBS)
            files = {p: str(p.relative_to(WORK)) for p in keep}
            sha = sync_results(REPO, auth_url(REPO_URL, SECRET), RUN, files, branch, {}, message='results: STNet ' + ', '.join(RUN_JOBS), secret=SECRET)
            print('pushed to', branch, sha)
        except Exception as e:
            print('push failed (the zip is still there):', clean(str(e))[:300])